In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import json
import ast
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import math
from plotly.subplots import make_subplots
import plotly.express as px
from langchain_openai import ChatOpenAI
import getpass
from bs4 import BeautifulSoup
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, validator
import dask.dataframe as dd
import faiss
import torch
from enum import Enum
from sentence_transformers import SentenceTransformer
import dask.dataframe as dd
import os
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from sklearn.metrics import precision_recall_fscore_support

In [ ]:
import re
import warnings
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
warnings.filterwarnings("ignore")

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    jaccard_score,
    classification_report  
)

### Getting the Data

In [ ]:
#Load the LabelStudio dataframe 
df1 = pd.read_csv('/content/sample_data/project-5-at-2026-02-23-13-50-a08b9dc1.csv')
print(f"DataFrame shape: {df1.shape}")

In [ ]:
cols = ["id", "clean_content_en_deepl", "content_en_llm", "hazard_type", "strategy_type"]
df2= df1[cols]
df2 = df2.drop_duplicates(subset="id", keep="first")   # or keep="last"

In [ ]:
### Load the dataset with all data ready for downstream tasks ###
file_path = '/content/drive/MyDrive/deep_learning_7_SV/Blatten_news/'
#file_name = 'news_predictions_all_relevant_translated_lemma_20251201.csv'
file_name = 'my_news_translated_en_LLM_final_ver20260105.csv'
df_orig = pd.read_csv(file_path + file_name)
print(f"DataFrame shape: {df_orig.shape}")

In [ ]:
df_orig = df_orig.drop_duplicates(subset="id", keep="first")   # or keep="last"
# Merge the two dataframes on the 'id' column
common_ids = set(df_orig["id"]) & set(df2["id"])
len(common_ids)
df3 = df_orig.copy()

df3 = df3.merge(
    df2[['id', 'hazard_type', 'strategy_type']],
    on='id',
    how='left'
)

In [ ]:
#Drop extra column that were generated by the SetFit classfier as they are not needed for the downstream tasks
df3 = df3.drop(columns=["label", "probability_0","probability_1","predicted_label"])

### Preparing the Data

In [ ]:
def extract_taxonomy(val):
    if pd.isna(val):
        return []

    try:
        # Convert string → Python object
        data = ast.literal_eval(val)

        result = []
        for item in data:
            for group in item.get("taxonomy", []):
                result.extend(group)   # flatten [["a"],["b"]] → ["a","b"]

        return result

    except Exception as e:
        print("Parsing error:", e)
  
        return []
    
# Apply the function to the 'hazard_type' column aand strategy_type column
df3["hazard_type"] = df3["hazard_type"].apply(extract_taxonomy)
df3["strategy_type"] = df3["strategy_type"].apply(extract_taxonomy)

In [ ]:
#Check the distribution of the hazard_type column
df3["hazard_type"].head()
df3["hazard_type"].str.len().value_counts()
df3[df3["hazard_type"].notna()]
df3["hazard_type"].apply(lambda x : len(x)).value_counts().to_frame().T
df3["hazard_type"].explode().value_counts()
df3["hazard_type"].apply(lambda x: len(x) > 0).sum()
df3["hazard_type"].apply(lambda x : len(x)).value_counts().to_frame().T

In [ ]:
df3["strategy_type"].explode().value_counts()
df3["strategy_type"].apply(lambda x : len(x)).value_counts().to_frame().T

In [ ]:
label_map_h = {"extreme temperature": "temperature",
             "heatwave": "temperature"}

label_map_s = {"civil protection\\/armed forces": "civil protection armed forces",
               "fundraising": "solidarity",
               "relocation (temporary)": "relocation",
               "relocation (permanent)":'relocation',
               "solidarity": "solidarity and donation",
               "donation": "solidarity and donation",
               "fundraising": "solidarity and donation",
               "emergency planning": "others",
               "medical support": "others",
               "(risk based) land use planning": "others"}

df3["hazard_type"] = df3["hazard_type"].apply(
    lambda xs: [label_map_h.get(x, x) for x in xs] if isinstance(xs, list) else xs
)

df3["strategy_type"] = df3["strategy_type"].apply(
    lambda xs: [label_map_s.get(x, x) for x in xs] if isinstance(xs, list) else xs
)

In [ ]:
df3["split_hazard"] = "unlabelled_hazard"
mask = df3["hazard_type"].apply(lambda x: len(x) > 0)
df3.loc[mask, "split_hazard"] = "labelled_hazard"
df3["split_hazard"].value_counts().to_frame()

In [ ]:
df3["split_strategy"] = "unlabelled_strategy"
mask = df3["strategy_type"].apply(lambda x: len(x)) > 0
df3.loc[mask, "split_strategy"] = "labelled_strategy"
df3["split_strategy"].value_counts().to_frame()

In [ ]:
(df3["content_en"].str.split().apply(len)
 .hist(bins=np.linspace(0, 2500, 50), grid=False, edgecolor="C0"))
plt.title("Words per news")
plt.xlabel("Number of words")
plt.ylabel("Number of news")
plt.show()

## Multilabel with LLM structured output

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

ChatOpenAI.api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
# Define prompt template for extracting structured information using the 'Data' schema.
tagging_prompt = ChatPromptTemplate.from_template(
    """
Extract the desired information from the following dataset.

Only extract the information mentioned in the 'Data' function.

Do not make up content. Rigorously extract only the required information.


Passage:
{input}
"""
)

In [ ]:
class EventEnum(str, Enum):
    avalanche = "avalanche"
    debris_flow = "debris flow"
    drought = "drought"
    extreme_precipitation = "extreme precipitation"
    flood = "flood"
    glacier_collapse = "glacier collapse"
    glacier_melting = "glacier melting"
    lake_formation = "lake formation"
    landslide = "landslide"
    mudslide = "mudslide"
    permafrost_melting = "permafrost melting"
    rockfall = "rockfall"
    storm = "storm"
    temperature = "temperature"

class StrategyEnum(str, Enum):
    camera_gps_radar_satellite = "camera, gps, radar, satellite"
    civil_protection_armed_forces = "civil protection armed forces"
    debris_clearing = "debris clearing"
    disaster_relief_fund = "disaster relief fund"
    early_warning = "early warning"
    emergency_aid = "emergency aid"
    emergency_law = "emergency law"
    evacuation = "evacuation"
    exclusion_area = "exclusion area"
    financial_aid = "financial aid"
    hazard_map = "hazard map"
    insurance = "insurance"
    monitoring = "monitoring"
    others = "others"
    preventive_measures = "preventive measures"
    reconnaissance_flights = "reconnaissance flights"
    reconstruction = "reconstruction"
    relocation = "relocation"
    restore_critical_infrastructures = "restore critical infrastructures"
    road_closure = "road closure"
    search_and_rescue = "search and rescue"
    solidarity_and_donation = "solidarity and donation"
    studies_and_research = "studies and research"
    walls_forests_dikes_embankements = "walls, forests, dikes, embankements"

In [ ]:
#This schema is to analyse multilingual news. In the case of Blatten, we first run an fully open vocabulary using the set up here. We then manually annotated 10% of the news
#This annotation exercise led to a refinement of the event and strategy extraction, which led to setting up a more targeted extraction with a vocab.

class General(BaseModel):
    event: List[EventEnum] = Field(
        default_factory=list,
        description=(
            "Select one or more event from the allowed list. "
            "Return as a list (no free text)."
        ),
    )
    strategy: List[StrategyEnum] = Field(
        default_factory=list,
        description=(
             "Select one or more strategy labels from the allowed list based only on information explicitly stated in the text. "
             "Return only a list of allowed labels, with no free text. "
            "Use a label only if it is clearly supported by the text. "
        "Do not guess or infer weakly implied strategies. "
        "If no label is clearly supported by the text, return an empty list. "

        "Important decision rules: "
        "'relocation' = moving to another location for temporary or permanent stay; "
        "do not use 'relocation' for evacuation alone if there is no mention of temporary or permanent accomodation or location to move to. "
        "'exclusion area' = an area is officially closed, restricted, cordoned off, or forbidden to enter; "
        "do not use it for general warnings without explicit access restriction. "
        "'others' = use only as a last resort when a strategy is clearly present but does not fit any other allowed label; "
        "do not use 'others' if another allowed label is a reasonable match. "
        ),
    )

class Data(BaseModel):
    """Extracted data about general information and topics."""
    # Creates a model so that we can extract multiple entities.
    general: List[General]

In [ ]:
llm = ChatOpenAI(temperature=0, model="gpt-5-nano-2025-08-07").with_structured_output(Data)
tagging_chain = tagging_prompt | llm  #make sure you refer to the correct prompt

In [ ]:
## Test extraction
inp2 = """
So far the mountain has withstood the pressure. The rock moves audibly.
On the ticker SRF News the authorities inform at o’clock about the
latest findings on the situation on the mountain in Blatten VS.
The mountain withstands another night the pressure. The tensions
in the ice continue, and the rock moves audibly. The movement of
the glacier in the landslide area of Blatten in the Valais Lötschental
amounts to roughly four to four and a half metres per day. Meanwhile
two thirds of the unstable area on the Kleine Nesthorn have detached.
Topics in this news ticker: press conference scheduled for o’clock,
federal government wants to monitor terrain movements nationwide
with satellites, Birch glacier continues movement down‑valley,
view on the mountain cleared again, fresh break‑offs visible.
The mountain endures another night, tension remains.
Regional command staff: mountain shows increasing movement.
Movement on the mountain remains visible. Clear night on the
Kleine Nesthorn: situation remains tense.
The two possible scenarios from Blatten authorities:
landslide with larger volumes rather unlikely.

"""
run = tagging_chain.invoke({"input": inp2})

# If a wrapper object is returned, unwrap it
if hasattr(run, "parsed") and run.parsed is not None:
    run = run.parsed  # now run should be Data

print(run.general[0].event)
print(run.general[0].strategy)

In [ ]:
output_file_path = '/content/sample_data/cleaned_Blatten.parquet'

# Check if the parquet file already exists
if os.path.exists(output_file_path):
    print(f"Parquest file already exists at {output_file_path}. Skipping processing...")
    ddf = dd.read_parquet(output_file_path)  # Load the existing cleaned dataset
else:
    print("Cleaned file not found. Processing the dataset...")
    print(df3.columns)

    # Save the cleaned and deduplicated dataset
    df3.to_parquet(output_file_path, engine='pyarrow')
    print(f"Cleaned and deduplicated dataset saved to {output_file_path}")

In [ ]:
# Define the path to save/load embeddings
embeddings_path = '/content/sample_data/embeddings_Blatten.npy'

# Check if embeddings already exist
if os.path.exists(embeddings_path):
    print("Embeddings file found. Loading embeddings...")
    embeddings_np = np.load(embeddings_path)
else:
    print("Embeddings file not found. Generating embeddings...")

    # Load the cleaned and deduplicated dataset
    cleaned_file_path = '/content/sample_data/cleaned_Blatten.parquet'
    ddf = dd.read_parquet(cleaned_file_path)
    df = ddf.compute()  # Convert to pandas DataFrame

    # Define a function to encode text in batches
    def encode_in_batches(texts, model, batch_size=64):
        """Encodes text in smaller batches to manage memory usage efficiently."""
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            embeddings.append(model.encode(batch, convert_to_tensor=True))
        return torch.cat(embeddings)

    # Load the model and encode the `content` column
    model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
    data = df3['content_en'].tolist()

    # Encode content in batches
    embeddings = encode_in_batches(data, model)

    # Convert embeddings to NumPy and save to disk
    embeddings_np = embeddings.cpu().numpy()
    np.save(embeddings_path, embeddings_np)
    print(f"Embeddings saved to {embeddings_path}")

In [ ]:
# File path for the output dataset
#output_parquet_path = '/content/drive/MyDrive/deep_learning_7_SV/task3/unique_dataset.parquet'
output_parquet_path = '/content/sample_data/unique_dataset_Blatten.parquet'

# Check if the unique dataset file already exists
if os.path.exists(output_parquet_path):
    print(f"Unique dataset already exists at {output_parquet_path}. Skipping FAISS processing...")
    unique_df = pd.read_parquet(output_parquet_path)  # Load the deduplicated dataset
else:
    print("Unique dataset not found. Running FAISS to deduplicate...")

    # Perform nearest neighbor search using FAISS
    # - FAISS helps identify near-duplicate rows using embeddings.
    # - L2 distance is used to measure similarity between embeddings.
    index = faiss.IndexFlatL2(embeddings_np.shape[1])  # Create FAISS index
    index.add(embeddings_np)  # Add embeddings to the FAISS index

    # Query for similar items
    # - Retrieves the top 2 nearest neighbors for each embedding.
    D, I = index.search(embeddings_np, 2)

    # Keep only the first occurrences based on a similarity threshold
    keep_mask = [True] * len(embeddings_np)  # Initialize mask to keep all rows
    for idx, (distances, indices) in enumerate(zip(D, I)):
        if distances[1] < 0.10:  # Adjust this threshold as needed
            keep_mask[indices[1]] = False  # Mark near-duplicates for removal

    # Calculate metrics for dropped rows
    total_rows = len(embeddings_np)
    kept_rows = sum(keep_mask)
    dropped_rows = total_rows - kept_rows
    dropped_percentage = (dropped_rows / total_rows) * 100

    # Log the metrics
    print(f"Total rows: {total_rows}")
    print(f"Rows kept: {kept_rows}")
    print(f"Rows dropped: {dropped_rows}")
    print(f"Percentage of rows dropped: {dropped_percentage:.2f}%")

    # Create a DataFrame with unique rows
    unique_df = df3[keep_mask]

    # Save the unique dataset to Parquet
    unique_df.to_parquet(output_parquet_path, engine='pyarrow', index=False)
    print(f"Unique dataset saved to {output_parquet_path}")

In [ ]:
def safe_tagging(content):
    try:
        res = tagging_chain.invoke({"input": content})

        # Unwrap if LangChain returns a wrapper with `.parsed`
        if hasattr(res, "parsed") and res.parsed is not None:
            res = res.parsed

        return res  # should be a Data object (Pydantic)

    except Exception as e:
        return {"error": str(e)}

def parallel_tagging(data, max_workers=8):
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(tqdm(executor.map(safe_tagging, data), total=len(data), desc="Tagging"))

results = parallel_tagging(unique_df["content_en"], max_workers=8)
print(f"Tagging completed for {len(results)} rows.")

def first_info(result):
    """Return first LaySummary or None (handles errors/dicts)."""
    if isinstance(result, dict) and "error" in result:
        return None
    if result is None or not getattr(result, "general", None):
        return None
    return result.general[0]

# Domain is a list[DomainEnum]; convert to "a; b; c"
unique_df["event_llm"] = [
    ("; ".join([d.value for d in first_info(r).event]) if first_info(r) and first_info(r).event else None)
    for r in results
]

unique_df["strategy_llm"] = [
    ("; ".join([d.value for d in first_info(r).strategy]) if first_info(r) and first_info(r).strategy else None)
    for r in results
]

In [ ]:
final_output_path = '/content/drive/MyDrive/deep_learning_7_SV/Blatten_news/Blatten_DataExtraction_GivenLabels_20260313.csv'
unique_df.to_csv(final_output_path, index=False)

# Log the save operation
print(f"Processed dataset saved to {final_output_path}")

In [ ]:
df_eval_hazard = unique_df[unique_df['split_hazard']=='labelled_hazard']

In [ ]:
#def normalize_labels(labels):
#    return [label_map.get(lab.strip().lower(), lab.strip().lower()) for lab in labels]

# parse predicted labels from string
def parse_llm_labels(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [lab.strip().lower() for lab in str(x).split(";") if lab.strip()]


df_eval_hazard["y_true"] = df_eval_hazard["hazard_type"].apply(parse_llm_labels)
df_eval_hazard["y_pred"] = df_eval_hazard["event_llm"].apply(parse_llm_labels)

In [ ]:
# collect all labels appearing in either truth or prediction
all_labels = sorted(set(
    lab for row in pd.concat([df_eval_hazard["y_true"], df_eval_hazard["y_pred"]]) for lab in row
))

mlb = MultiLabelBinarizer(classes=all_labels)
Y_true = mlb.fit_transform(df_eval_hazard["y_true"])
Y_pred = mlb.transform(df_eval_hazard["y_pred"])

print("Micro F1:  ", f1_score(Y_true, Y_pred, average="micro", zero_division=0))
print("Macro F1:  ", f1_score(Y_true, Y_pred, average="macro", zero_division=0))
print("Samples F1:", f1_score(Y_true, Y_pred, average="samples", zero_division=0))

print("Micro Precision:", precision_score(Y_true, Y_pred, average="micro", zero_division=0))
print("Micro Recall:   ", recall_score(Y_true, Y_pred, average="micro", zero_division=0))

print("Exact match accuracy:", accuracy_score(Y_true, Y_pred))
print("Jaccard score:", jaccard_score(Y_true, Y_pred, average="samples", zero_division=0))

In [ ]:
p, r, f1, s = precision_recall_fscore_support(
    Y_true, Y_pred, average=None, zero_division=0
)

results = pd.DataFrame({
    "label": mlb.classes_,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support": s
}).sort_values("f1")

print(results)

In [ ]:
df_eval_strategy = unique_df[unique_df['split_strategy']=='labelled_strategy']

In [ ]:
df_eval_strategy["y_true"] = df_eval_strategy["strategy_type"].apply(parse_labels)
df_eval_strategy["y_pred"] = df_eval_strategy["strategy_llm"].apply(parse_llm_labels)

# collect all labels appearing in either truth or prediction
all_labels = sorted(set(
    lab for row in pd.concat([df_eval_strategy["y_true"], df_eval_strategy["y_pred"]]) for lab in row
))

mlb = MultiLabelBinarizer(classes=all_labels)
Y_true = mlb.fit_transform(df_eval_strategy["y_true"])
Y_pred = mlb.transform(df_eval_strategy["y_pred"])

print("Micro F1:  ", f1_score(Y_true, Y_pred, average="micro", zero_division=0))
print("Macro F1:  ", f1_score(Y_true, Y_pred, average="macro", zero_division=0))
print("Samples F1:", f1_score(Y_true, Y_pred, average="samples", zero_division=0))

print("Micro Precision:", precision_score(Y_true, Y_pred, average="micro", zero_division=0))
print("Micro Recall:   ", recall_score(Y_true, Y_pred, average="micro", zero_division=0))

print("Exact match accuracy:", accuracy_score(Y_true, Y_pred))
print("Jaccard score:", jaccard_score(Y_true, Y_pred, average="samples", zero_division=0))

p, r, f1, s = precision_recall_fscore_support(
    Y_true, Y_pred, average=None, zero_division=0
)

results_s = pd.DataFrame({
    "label": mlb.classes_,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support": s
}).sort_values("f1")

print(results_s)

In [ ]:
plot_df = results.sort_values("f1")
plt.figure(figsize=(8, 5))
plt.barh(plot_df["label"], plot_df["f1"])
plt.xlabel("F1 score")
plt.ylabel("")
plt.title("Per-label F1 scores")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = results_s.sort_values("f1")
plt.figure(figsize=(8, 5))
plt.barh(plot_df["label"], plot_df["f1"])
plt.xlabel("F1 score")
plt.ylabel("")
plt.title("Per-label F1 scores")
plt.tight_layout()
plt.show()